# Making the AD_KG_CD_AF collection

In [1]:
%store -r

In [2]:
import commute_dm.bel_export
import commute_dm.queries
import commute_dm.submaps
import credentials
import momapy.io.core
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)
commute_dm.queries.prewarm_session(session)

In [4]:
BEL_COLLECTION_NAME = "AD_KG_BEL"
CD_COLLECTION_NAME = "AD_KG_CD_AF"
CD_AF_SOURCE_COLLECTION_NAMES = [
    "COVID_DM_CD",
    "COVID_DM_CD_AF",
    "PD_DM_CD",
    "PD_DM_CD_AF",
]
OUTPUT_FILE_PATH = AD_KG_CD_AF_BUILD_DIR / "ad_kg.xml"
OUTPUT_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)

## Transforming

In [5]:
projection = commute_dm.bel_export.get_bel_influence_graph_projection(
    session, [BEL_COLLECTION_NAME]
)
cd_map, element_to_annotations = (
    commute_dm.bel_export.make_cd_map_from_bel_influence_graph_projection(
        session, [BEL_COLLECTION_NAME], projection
    )
)

## Storing

In [6]:
cd_map = commute_dm.submaps.renumber_ids(cd_map)
element_to_annotations = {
    species: annotations
    for species in commute_dm.submaps.iter_species_and_subunits(cd_map.model.species)
    if (annotations := element_to_annotations.get(species))
}
_ = momapy.io.core.write(
    cd_map,
    OUTPUT_FILE_PATH,
    writer="celldesigner",
    element_to_annotations=element_to_annotations,
)

In [7]:
SEED_QUERY = """
MATCH (collection:Collection)-[:HAS_ENTRY]->()-[:HAS_OBJ]->(:CellDesignerMap)
      -[:HAS_MODEL]->(:Model)-[:HAS_MODEL_ELEMENT]->(model_element)
WHERE collection.name IN $collection_names
RETURN DISTINCT model_element
"""

object_key_to_node = {}
_ = session.execute_query_as_objects(
    SEED_QUERY,
    params={"collection_names": CD_AF_SOURCE_COLLECTION_NAMES},
    node_id_to_object={},
    object_key_to_node=object_key_to_node,
)
_ = session.save_collections_from_file_paths(
    [(CD_COLLECTION_NAME, [OUTPUT_FILE_PATH])],
    return_type="map",
    with_membership_edges=True,
    integration_mode="hash",
    object_key_to_node=object_key_to_node,
)